# Pareja B · Idea 3 — Cabeza con capas ocultas (MLP)

Una capa lineal solo traza fronteras rectas. Un MLP aprende combinaciones ("tonos oscuros + una figura + ciudad de noche = Misterio"). Rejilla: 1 a 3 capas ocultas, ancho 256/512/1024, ReLU/GELU, dropout 0.2/0.4, lr 1e-3/3e-4, LayerNorm sí/no. Un MLP 512→512→17 tiene ~271 mil parámetros. Empiecen sin esperar a la Pareja A, con los vectores B/32.

**Dónde se programa:** `cabezas.py`: claves `activacion: gelu` y `layernorm: true`. La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** su prueba pasa (quiten el `xfail` de `test_idea3` en
`tests/test_pendientes.py`) y el resultado en validación está en el registro.

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).

## Preparar el entorno

**CPU basta**: entrenar una cabeza con vectores ya calculados tarda segundos.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`B-cabezas`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "B-cabezas"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt

In [ ]:
# ¿ya pasa la prueba de la idea? (mientras esté pendiente sale como "xfailed")
!python -m pytest -q tests/test_pendientes.py -k idea3 -rxX

## 1. Rejilla

Cada configuración se guarda como `experimentos/B3_<...>.yaml` (se suben todas: aunque no mejore,
se registra). Las que usan GELU o LayerNorm fallan hasta que la idea 3 esté en `cabezas.py`.

In [ ]:
import itertools, json, pathlib

OCULTAS = [[512], [512, 256], [1024, 512, 256]]
ACTIVACION = ["relu"]           # agreguen "gelu" cuando esté implementada
DROPOUT = [0.2, 0.4]
TASA = [1e-3, 3e-4]
LAYERNORM = [False]             # agreguen True cuando esté implementada

nombres = []
for ocultas, act, drop, tasa, ln in itertools.product(OCULTAS, ACTIVACION, DROPOUT, TASA, LAYERNORM):
    nombre = f"B3_{'x'.join(map(str, ocultas))}_{act}_d{drop}_lr{tasa:g}" + ("_ln" if ln else "")
    cabeza = {"tipo": "mlp", "ocultas": ocultas, "dropout": drop}
    if act != "relu":
        cabeza["activacion"] = act
    if ln:
        cabeza["layernorm"] = True
    pathlib.Path(f"experimentos/{nombre}.yaml").write_text(
        f"nombre: {nombre}\npareja: B\ndescripcion: Idea 3, rejilla de MLP.\n"
        f"vectores: [clip_b32__centro]\ncabeza: {json.dumps(cabeza)}\n"
        f"entrenamiento: {{tasa: {tasa}}}\n")
    nombres.append(nombre)
print(len(nombres), "configuraciones")

In [ ]:
for nombre in nombres:
    !python -m clasificador.entrenar experimentos/{nombre}.yaml
    !python -m clasificador.evaluar {nombre}

## 2. Las mejores en validación (y luego `comparar base <la mejor>`)

In [ ]:
import pandas as pd
!python -m clasificador.evaluar --registro > /dev/null
registro = pd.read_csv("resultados/registro.csv")
registro[registro["nombre"].str.startswith("B3_")].sort_values("F1_macro", ascending=False).head(10)

## Subir a la rama `B-cabezas`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`B-cabezas`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")